# learn-to-remember: M4 GRPO training (Kaggle T4 / any CUDA GPU)

Run **after M3**: `CHOSEN` must be the model the M3 probe selected (docs/DECISIONS.md).
Order (spec §7, §5): 1) measure s/step on the main config, 2) overfit sanity check (8 questions, reward should reach ~1),
3) budget table from the measured numbers, 4) main run (experiment 2: notes v3, lambda 0.2, 200 steps), 5) eval of the trained adapter.

Kaggle sessions stop at 12 h. The main run checkpoints every 10 steps; to continue in a new session, add this notebook's
previous output as an input and the resume cell copies the checkpoint back. If the budget table says the run needs
many sessions, use a rented GPU instead (`scripts/run_m4.sh`).

In [ ]:
import os, subprocess, json, glob
REPO = "/kaggle/working/learn-to-remember"
if not os.path.exists(REPO):
    if os.path.exists("/kaggle/input/learn-to-remember"):
        subprocess.run(["cp", "-r", "/kaggle/input/learn-to-remember", REPO], check=True)
    else:
        subprocess.run(["git", "clone", "https://github.com/Vivek-Chaudhari30/learn-to-remember.git", REPO], check=True)
os.chdir(REPO)
print(subprocess.run(["git", "log", "--oneline", "-3"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q "transformers>=4.51,<6" "peft>=0.12" "accelerate>=0.34" pyyaml
from transformers import AutoTokenizer
for m in ["Qwen/Qwen2.5-0.5B-Instruct", "Qwen/Qwen3-1.7B"]:
    AutoTokenizer.from_pretrained(m)
!python -m pytest -q 2>&1 | tail -3

In [ ]:
CHOSEN = "Qwen/Qwen2.5-1.5B-Instruct"   # <- set from the M3 probe
DTYPE = "fp16"                           # T4: fp16 (+ GradScaler, automatic). A100/L4: bf16
MICRO = 2                                # lower to 1 if the update OOMs
CKPT = "/kaggle/working/checkpoints"

## 1. Seconds per step on the main config (3 steps, no checkpoint)

In [ ]:
!python -u scripts/train.py --config configs/train_notes.yaml --model {CHOSEN} --dtype {DTYPE} --micro-batch {MICRO} --steps 3 --timing --out results/train/timing 2>&1 | grep -v -i warn

## 2. Overfit sanity check (8 fixed questions, 60 steps)

In [ ]:
!python -u scripts/train.py --config configs/overfit.yaml --model {CHOSEN} --dtype {DTYPE} --micro-batch {MICRO} --out results/train/overfit --save-dir {CKPT}/overfit 2>&1 | grep -v -i warn

In [ ]:
h = json.load(open("results/train/overfit/train_log.json"))
print("reward by step:", [round(x["reward_mean"], 3) for x in h])
print("first 5 mean %.3f -> last 5 mean %.3f" % (sum(x["reward_mean"] for x in h[:5]) / 5, sum(x["reward_mean"] for x in h[-5:]) / 5))

## 3. Budget table (from the measured s/step above)

In [ ]:
t = json.load(open("results/train/timing/timing.json"))
s = t["mean_sec_per_step"]
print(f"measured: {s:.1f} s/step (rollout {t['mean_sec_rollout']:.1f}, ref+update {t['mean_sec_update']:.1f}) on {t['gpu']}, P={t['P']} G={t['G']}, peak {t.get('peak_gpu_gb')} GB")
for steps in (50, 100, 200):
    print(f"  {steps} steps -> {steps * s / 3600:.1f} h  ({steps * s / 3600 / 11.5:.1f} Kaggle sessions of 11.5 h)")

## 4. Main run (experiment 2): notes v3, lambda 0.2, 200 steps, resumable

To continue from a previous session: add the previous version's output as an input; the next cell copies its checkpoint.

In [ ]:
prev = glob.glob("/kaggle/input/*/checkpoints/notes_l0.2/training_state.pt")
if prev and not os.path.exists(f"{CKPT}/notes_l0.2/training_state.pt"):
    src = os.path.dirname(prev[0]); os.makedirs(CKPT, exist_ok=True)
    subprocess.run(["cp", "-r", src, f"{CKPT}/notes_l0.2"], check=True)
    for f in glob.glob(os.path.join(os.path.dirname(os.path.dirname(os.path.dirname(prev[0]))), "learn-to-remember/results/train/notes_l0.2/*")):
        os.makedirs("results/train/notes_l0.2", exist_ok=True); subprocess.run(["cp", "-r", f, "results/train/notes_l0.2/"])
    print("resuming from", src)

In [ ]:
!python -u scripts/train.py --config configs/train_notes.yaml --model {CHOSEN} --dtype {DTYPE} --micro-batch {MICRO} --out results/train/notes_l0.2 --save-dir {CKPT}/notes_l0.2 --resume 2>&1 | grep -v -i warn

## 5. Evaluate the trained adapter (company A, notes v3; greedy, same harness as the M3 baselines)

In [ ]:
!python -u scripts/run_eval.py --model {CHOSEN} --adapter {CKPT}/notes_l0.2 --dtype {DTYPE} --companies company_a --conditions v3 --out results/trained/notes_l0.2 2>&1 | grep -v -i warn | tail -12

## Package

In [ ]:
!zip -qr /kaggle/working/m4_results.zip results/train results/trained && ls -lh /kaggle/working/m4_results.zip